<a href="https://colab.research.google.com/github/biglalo104/Projects/blob/main/Image%20Classification%20Using%20CNN%20on%20CIFAR10%20Dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import argparse
import os
import random
import time

In [6]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as T
from torch.utils.data import DataLoader, Dataset
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

In [7]:

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [4]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as T
from torch.utils.data import DataLoader, Dataset
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
)
import sys
import argparse
import os
import random
import time

CLASSES = (
    "plane",
    "car",
    "bird",
    "cat",
    "deer",
    "dog",
    "frog",
    "horse",
    "ship",
    "truck",
)


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


class TransformSubset(Dataset):
    """
    Utility dataset to apply a specific transform to a subset of a base dataset.
    This allows using the original CIFAR-10 training set for both training and
    validation while applying training augmentation only to the training split.
    """

    def __init__(self, base_dataset, indices, transform=None):
        self.base_dataset = base_dataset
        self.indices = indices
        self.transform = transform

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        img, label = self.base_dataset[self.indices[idx]]
        if self.transform is not None:
            img = self.transform(img)
        return img, label


class SimpleCNN(nn.Module):
    """
    Compact CNN for CIFAR-10.

    Architecture:
        Block 1: 2 conv layers, 32 filters
        Block 2: 2 conv layers, 64 filters
        Block 3: 2 conv layers, 128 filters
        Classifier: FC -> ReLU -> Dropout -> FC
    """

    def __init__(self, num_classes=10):
        super().__init__()

        self.features = nn.Sequential(
            # Block 1
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.2),
            # Block 2
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.3),
            # Block 3
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.4),
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


def get_dataloaders(args):
    train_transform = T.Compose(
        [
            T.RandomCrop(32, padding=4),
            T.RandomHorizontalFlip(),
            T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
            T.ToTensor(),
            T.Normalize(
                mean=(0.4914, 0.4822, 0.4465),
                std=(0.2470, 0.2435, 0.2616),
            ),
        ]
    )

    eval_transform = T.Compose(
        [
            T.ToTensor(),
            T.Normalize(
                mean=(0.4914, 0.4822, 0.4465),
                std=(0.2470, 0.2435, 0.2616),
            ),
        ]
    )

    base_train = torchvision.datasets.CIFAR10(
        root=args.data,
        train=True,
        download=True,
        transform=None,
    )

    test_set = torchvision.datasets.CIFAR10(
        root=args.data,
        train=False,
        download=True,
        transform=eval_transform,
    )

    n = len(base_train)
    indices = list(range(n))

    rng = np.random.default_rng(args.seed)
    rng.shuffle(indices)

    val_size = int(n * args.val_split)
    train_indices = indices[val_size:]
    val_indices = indices[:val_size]

    train_set = TransformSubset(base_train, train_indices, train_transform)
    val_set = TransformSubset(base_train, val_indices, eval_transform)

    loader_kwargs = {
        "num_workers": args.workers,
        "pin_memory": True,
    }

    train_loader = DataLoader(
        train_set,
        batch_size=args.batch_size,
        shuffle=True,
        **loader_kwargs,
    )

    val_loader = DataLoader(
        val_set,
        batch_size=args.batch_size,
        shuffle=False,
        **loader_kwargs,
    )

    test_loader = DataLoader(
        test_set,
        batch_size=args.batch_size,
        shuffle=False,
        **loader_kwargs,
    )

    return train_loader, val_loader, test_loader


def train_epoch(model, loader, criterion, optimizer, device):
    model.train()

    total_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        preds = outputs.argmax(dim=1)

        total_loss += loss.item() * images.size(0)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    avg_loss = total_loss / total
    accuracy = correct / total

    return avg_loss, accuracy


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()

    all_preds = []
    all_labels = []

    total_loss = 0.0
    total = 0

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        outputs = model(images)
        loss = criterion(outputs, labels)
        preds = outputs.argmax(dim=1)

        total_loss += loss.item() * images.size(0)
        total += labels.size(0)

        all_preds.append(preds.cpu())
        all_labels.append(labels.cpu())

    all_preds = torch.cat(all_preds).numpy()
    all_labels = torch.cat(all_labels).numpy()

    accuracy = accuracy_score(all_labels, all_preds)
    macro_precision = precision_score(
        all_labels,
        all_preds,
        average="macro",
        zero_division=0,
    )
    macro_recall = recall_score(
        all_labels,
        all_preds,
        average="macro",
        zero_division=0,
    )
    macro_f1 = f1_score(
        all_labels,
        all_preds,
        average="macro",
        zero_division=0,
    )

    report = classification_report(
        all_labels,
        all_preds,
        target_names=CLASSES,
        digits=4,
        zero_division=0,
    )

    cm = confusion_matrix(all_labels, all_preds)

    return {
        "loss": total_loss / total,
        "accuracy": float(accuracy),
        "macro_precision": float(macro_precision),
        "macro_recall": float(macro_recall),
        "macro_f1": float(macro_f1),
        "classification_report": report,
        "confusion_matrix": cm.tolist(),
    }


def generate_report(args, test_metrics, best_val_acc, history):
    cm = test_metrics["confusion_matrix"]

    cm_header = "| true/pred | " + " | ".join(CLASSES) + " |"
    cm_sep = "|---" * (len(CLASSES) + 1) + "|"
    cm_rows = []

    for i, row in enumerate(cm):
        cm_rows.append(
            f"| {CLASSES[i]} | " + " | ".join(str(x) for x in row) + " |"
        )

    cm_table = "\n".join([cm_header, cm_sep] + cm_rows)

    if history:
        hist_rows = []
        for h in history[-10:]:
            hist_rows.append(
                f"| {h['epoch']} | {h['train_loss']:.4f} | {h['train_acc']:.4f} | "
                f"{h['val_loss']:.4f} | {h['val_acc']:.4f} |"
            )
        hist_table = "\n".join(hist_rows)
    else:
        hist_table = "No training history recorded."

    report = f"""# CNN Image Classification Report: CIFAR-10

## 1. Objective

Build and evaluate a convolutional neural network for image classification on the CIFAR-10 dataset.

## 2. Dataset

- Dataset: CIFAR-10
- Image size: 32 x 32 RGB
- Number of classes: 10
- Original training set size: 50,000 images
- Original test set size: 10,000 images
- Training/validation split used:
  - Training subset: {int((1 - args.val_split) * 50000)} images
  - Validation subset: {int(args.val_split * 50000)} images

The validation split was used for model selection. The test set was used only once for final evaluation.

## 3. Model

The model is a compact CNN trained from scratch.

Main components:

- Six convolutional layers.
- Batch normalization after each convolution.
- Max pooling after each convolutional block.
- Dropout layers to reduce overfitting.
- Fully connected classification head.

## 4. Training Configuration

- Loss function: CrossEntropyLoss
- Optimizer: Adam
- Learning rate: {args.lr}
- Weight decay: {args.weight_decay}
- Batch size: {args.batch_size}
- Epochs: {args.epochs}
- Learning-rate scheduler: CosineAnnealingLR
- Data augmentation:
  - Random crop with padding
  - Random horizontal flip
  - Color jitter
- Normalization:
  - CIFAR-10 channel mean and standard deviation

## 5. Final Test Performance

| Metric | Value |
|---|---:|
| Test accuracy | {test_metrics['accuracy']:.4f} |
| Macro precision | {test_metrics['macro_precision']:.4f} |
| Macro recall | {test_metrics['macro_recall']:.4f} |
| Macro F1-score | {test_metrics['macro_f1']:.4f} |
| Test loss | {test_metrics['loss']:.4f} |
| Best validation accuracy | {best_val_acc:.4f} |

## 6. Classification Report

```text
{test_metrics['classification_report']}
```

## 7. Confusion Matrix

Rows are true labels. Columns are predicted labels.

{cm_table}

## 8. Last 10 Training Epochs

| Epoch | Train loss | Train accuracy | Val loss | Val accuracy |
|---|---:|---:|---:|---:|
{hist_table}

## 9. Notes

This report was generated automatically by `train_cifar10_cnn.py`.
"""

    return report


def parse_args():
    parser = argparse.ArgumentParser(
        description="Train and evaluate a CNN on CIFAR-10."
    )

    parser.add_argument("--data", type=str, default="./data")
    parser.add_argument("--epochs", type=int, default=50)
    parser.add_argument("--batch-size", type=int, default=128)
    parser.add_argument("--lr", type=float, default=1e-3)
    parser.add_argument("--weight-decay", type=float, default=1e-4)
    parser.add_argument("--val-split", type=float, default=0.1)
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--workers", type=int, default=2)
    parser.add_argument(
        "--checkpoint",
        type=str,
        default="best_cifar10_cnn.pth",
    )
    parser.add_argument(
        "--report",
        type=str,
        default="model_report.md",
    )

    # In interactive environments like Colab/Jupyter, argparse can receive unexpected
    # arguments from the kernel (e.g., -f). We explicitly pass an empty list
    # to parse_args() to prevent this.
    # A more robust check for Colab/Jupyter environment is to check sys.argv[0].
    if 'ipykernel_launcher.py' in sys.argv[0] or 'colab_kernel_launcher.py' in sys.argv[0]:
        args = parser.parse_args([])
    else:
        args = parser.parse_args()

    return args


def main():
    args = parse_args()

    set_seed(args.seed)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    train_loader, val_loader, test_loader = get_dataloaders(args)

    model = SimpleCNN(num_classes=10).to(device)

    criterion = nn.CrossEntropyLoss()

    optimizer = optim.Adam(
        model.parameters(),
        lr=args.lr,
        weight_decay=args.weight_decay,
    )

    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=args.epochs,
    )

    best_val_acc = 0.0
    history = []

    start_time = time.time()

    for epoch in range(1, args.epochs + 1):
        train_loss, train_acc = train_epoch(
            model,
            train_loader,
            criterion,
            optimizer,
            device,
        )

        val_metrics = evaluate(
            model,
            val_loader,
            criterion,
            device,
        )

        scheduler.step()

        history.append(
            {
                "epoch": epoch,
                "train_loss": train_loss,
                "train_acc": train_acc,
                "val_loss": val_metrics["loss"],
                "val_acc": val_metrics["accuracy"],
            }
        )

        improved = val_metrics["accuracy"] > best_val_acc

        if improved:
            best_val_acc = val_metrics["accuracy"]
            torch.save(model.state_dict(), args.checkpoint)

        print(
            f"Epoch {epoch:03d}/{args.epochs} | "
            f"Train Loss {train_loss:.4f} | Train Acc {train_acc:.4f} | "
            f"Val Loss {val_metrics['loss']:.4f} | Val Acc {val_metrics['accuracy']:.4f}"
            + (" | saved" if improved else "")
        )

    training_minutes = (time.time() - start_time) / 60.0
    print(f"Training time: {training_minutes:.2f} minutes")

    if os.path.exists(args.checkpoint):
        model.load_state_dict(
            torch.load(args.checkpoint, map_location=device)
        )

    test_metrics = evaluate(
        model,
        test_loader,
        criterion,
        device,
    )

    report = generate_report(
        args,
        test_metrics,
        best_val_acc,
        history,
    )

    with open(args.report, "w", encoding="utf-8") as f:
        f.write(report)

    print("\nFinal test metrics:")
    print(f"Accuracy: {test_metrics['accuracy']:.4f}")
    print(f"Macro precision: {test_metrics['macro_precision']:.4f}")
    print(f"Macro recall: {test_metrics['macro_recall']:.4f}")
    print(f"Macro F1: {test_metrics['macro_f1']:.4f}")
    print(f"Report saved to: {args.report}")


if __name__ == "__main__":
    main()

Using device: cpu


100%|██████████| 170M/170M [20:24<00:00, 139kB/s]
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch 001/50 | Train Loss 1.8358 | Train Acc 0.3108 | Val Loss 1.4667 | Val Acc 0.4548 | saved


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch 002/50 | Train Loss 1.5834 | Train Acc 0.4126 | Val Loss 1.2560 | Val Acc 0.5188 | saved


...

Epoch 049/50 | Train Loss 0.5577 | Train Acc 0.8100 | Val Loss 0.4283 | Val Acc 0.8528


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch 050/50 | Train Loss 0.5596 | Train Acc 0.8108 | Val Loss 0.4273 | Val Acc 0.8560
Training time: 259.72 minutes


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)



Final test metrics:
Accuracy: 0.8526
Macro precision: 0.8529
Macro recall: 0.8526
Macro F1: 0.8515
Report saved to: model_report.md
